# 01 - Explore the measurements

Calibration starts with data, not with a model. This notebook opens the
measurement file that ships with the repository and answers three questions:

1. What is in it, and in which units?
2. Where is it broken, outliers, gaps, wrong sampling rate?
3. How do I cut out the window I actually want to calibrate on?

No simulation runs here, so everything below is instant.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from pyadm1ode_calibration import MeasurementData

# The notebooks live in notebooks/, the data one level up.
DATA = Path("..") / "data" / "plant_measurements.csv"

measurements = MeasurementData.from_csv(DATA)
print(f"{len(measurements.data)} rows, {measurements.data.index[0]} .. {measurements.data.index[-1]}")
measurements.data.head()

## What the columns mean

`MeasurementData` keeps a plain pandas frame in `.data`, indexed by timestamp.
Three groups matter for calibration:

| Group | Columns | Role |
| --- | --- | --- |
| Substrate feed | `Q_sub_maize`, `Q_sub_manure`, `Q_sub_grass` | **input** - drives the simulation |
| Gas side | `Q_gas`, `Q_ch4`, `Q_co2`, `CH4_content` | **target** - what the model should reproduce |
| Process state | `pH`, `VFA`, `TAC`, `FOS_TAC`, `T_digester` | target, and an early warning of trouble |

`summary()` gives the usual descriptive statistics in one call.

In [ ]:
measurements.summary()

## Look before you calibrate

A plot of the raw channels tells you more in five seconds than any statistic.
Watch the gas production: it follows the feed, but not perfectly.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)

measurements.data[["Q_sub_maize", "Q_sub_manure"]].plot(ax=axes[0])
axes[0].set_ylabel("feed [m3/d]")

measurements.data["Q_gas"].plot(ax=axes[1], color="tab:green")
axes[1].set_ylabel("Q_gas [m3/d]")

measurements.data["pH"].plot(ax=axes[2], color="tab:red")
axes[2].set_ylabel("pH [-]")
axes[2].set_xlabel("")

fig.suptitle("Raw measurements")
fig.tight_layout()

## Outliers and gaps

Real archives contain sensor spikes and missing stretches. Both distort a
calibration: the optimiser chases a spike, or silently fits fewer points than
you think.

`remove_outliers()` replaces suspicious values with NaN and returns how many it
found. `fill_gaps()` then closes the holes. Both work in place, so we copy the
object first to keep the raw version for comparison.

In [ ]:
import copy

cleaned = copy.deepcopy(measurements)

n_outliers = cleaned.remove_outliers(columns=["Q_gas", "pH"], method="zscore", threshold=3.0)
print(f"outliers flagged: {n_outliers}")
print(f"NaNs before filling: {int(cleaned.data[['Q_gas', 'pH']].isna().sum().sum())}")

cleaned.fill_gaps(columns=["Q_gas", "pH"], method="interpolate")
print(f"NaNs after filling:  {int(cleaned.data[['Q_gas', 'pH']].isna().sum().sum())}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
measurements.data["Q_gas"].plot(ax=ax, label="raw", alpha=0.5)
cleaned.data["Q_gas"].plot(ax=ax, label="cleaned")
ax.set_ylabel("Q_gas [m3/d]")
ax.legend()
ax.set_title("Effect of outlier removal and gap filling")
fig.tight_layout()

## Pick a window, pick a rate

Two more operations you will use in every calibration:

- `get_time_window()` returns a **new** `MeasurementData` for a slice - this is
  how a train/test split is built (notebook 05).
- `resample()` changes the sampling rate **in place**. Hourly data over a long
  period makes the simulation slow for no benefit; daily means are often enough.

In [ ]:
week = measurements.get_time_window("2024-01-01", "2024-01-08")
print(f"window: {len(week.data)} rows ({week.data.index[0]} .. {week.data.index[-1]})")

daily = copy.deepcopy(week)
daily.resample("1D", aggregation="mean")
print(f"resampled to daily: {len(daily.data)} rows")

daily.data[["Q_gas", "Q_ch4", "P_el"]].round(1)